# Notebook 07: Triplet Progression Simulation — Dynamic Phase Evolution

## Objectives

1. **Simulate the full triplet progression** Π⁽⁰⁾ → Π⁽¹⁾ → Π⁽²⁾ → Π⁽³⁾ over time
2. **Visualize phase transitions** as a claim evolves through different stability zones
3. **Demonstrate collapse at 45°** — the critical IA/ZOS boundary
4. **Show recovery paths** — how a claim can return from collapse to VC/GOS
5. **Provide parameter exploration** — drift rates, recovery rates, noise levels

## Core Framework

> **Triplet Progression:** Π⁽⁰⁾ expand → Π⁽¹⁾ extend → Π⁽²⁾ resist → Π⁽³⁾ synthesis
>
> At **45°**, the claim collapses from VC/GOS into IA/ZOS.

## Reference

This notebook integrates and extends concepts from all previous notebooks:
- `00-06` — Foundation, math claims, equations, embeddings, agents, phase lock demo

## 1. Setup and Core Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import sys
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Colab setup
if 'google.colab' in sys.modules:
    !pip install numpy matplotlib ipywidgets > /dev/null 2>&1
    from IPython.display import display, HTML, clear_output
    from IPython.display import Video
    print("✓ Colab environment ready")
else:
    print("✓ Local environment ready")

# Core functions (from previous notebooks)
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def angle_degrees(a: np.ndarray, b: np.ndarray) -> float:
    cos = cosine_similarity(a, b)
    return np.arccos(np.clip(cos, -1.0, 1.0)) * 180 / np.pi

def get_phase(angle: float) -> str:
    """Return the triplet phase for a given angle."""
    if angle < 10:
        return "Π⁽⁰⁾"
    elif angle < 35:
        return "Π⁽¹⁾"
    elif angle <= 55:
        return "Π⁽²⁾"
    else:
        return "Π⁽³⁾"

def lock_status(angle: float) -> Dict[str, object]:
    if angle < 10:
        return {"status": "VC / GOS", "phase": "Π⁽⁰⁾ expand", "interpretation": "Valid Construction — stable", 
                "color": "#2ecc71", "signal_noise": "Signal >> Noise", "emoji": "🔒✅"}
    elif angle < 35:
        return {"status": "weak VC", "phase": "Π⁽¹⁾ extend", "interpretation": "Partially constructed — monitor", 
                "color": "#f1c40f", "signal_noise": "Signal > Noise", "emoji": "⚠️"}
    elif angle <= 55:
        return {"status": "IA / ZOS", "phase": "Π⁽²⁾ resist", "interpretation": "Invalid Assignment — collapsing", 
                "color": "#e74c3c", "signal_noise": "Signal ≈ Noise", "emoji": "🔴💀"}
    else:
        return {"status": "weak IA / broken", "phase": "Π⁽³⁾ synthesis", "interpretation": "Decoupled — needs rebuild", 
                "color": "#95a5a6", "signal_noise": "Noise > Signal", "emoji": "⚰️"}

print("✓ Core functions loaded")
print("\n" + "=" * 60)
print("TRIPLET PROGRESSION SIMULATION — Dynamic Phase Evolution")
print("=" * 60)

## 2. Claim Evolution Simulator

A class that simulates a claim's angle evolution over time with configurable drift and recovery.

In [ ]:
@dataclass
class ClaimEvolution:
    """Simulate a claim's angle evolution through triplet phases."""
    
    initial_angle: float = 0.0
    drift_rate: float = 2.0        # degrees per time step
    noise_amplitude: float = 1.0   # random noise
    recovery_rate: float = 3.0     # recovery speed when active
    
    history: List[Dict] = field(default_factory=list)
    
    def step(self, time_step: int, force_recovery: bool = False) -> Dict:
        """Advance simulation by one time step."""
        if not self.history:
            current_angle = self.initial_angle
        else:
            current_angle = self.history[-1]['angle']
        
        # Apply dynamics
        if force_recovery or (current_angle > 60 and not force_recovery):
            # Recovery mode: angle decreases
            delta = -self.recovery_rate
        else:
            # Normal drift mode: angle increases
            delta = self.drift_rate
        
        # Add noise
        noise = np.random.randn() * self.noise_amplitude
        
        # Update angle, clamp to [0, 90]
        new_angle = current_angle + delta + noise
        new_angle = max(0, min(90, new_angle))
        
        # Get status
        status = lock_status(new_angle)
        phase = get_phase(new_angle)
        
        record = {
            'time': time_step,
            'angle': new_angle,
            'phase': phase,
            'status': status['status'],
            'emoji': status['emoji'],
            'color': status['color'],
            'is_critical': 35 <= new_angle <= 55,
            'collapsed': new_angle >= 45
        }
        self.history.append(record)
        return record
    
    def run(self, n_steps: int, recovery_at: Optional[List[int]] = None) -> List[Dict]:
        """Run simulation for n_steps, optionally applying recovery at specific times."""
        self.history = []
        recovery_at = recovery_at or []
        
        for t in range(n_steps):
            force_recovery = t in recovery_at
            self.step(t, force_recovery=force_recovery)
        
        return self.history
    
    def get_summary(self) -> Dict:
        """Get summary statistics of the simulation."""
        if not self.history:
            return {'error': 'No simulation data'}
        
        angles = [h['angle'] for h in self.history]
        phases = [h['phase'] for h in self.history]
        
        return {
            'final_angle': angles[-1],
            'final_phase': phases[-1],
            'final_status': self.history[-1]['status'],
            'max_angle': max(angles),
            'min_angle': min(angles),
            'time_to_45': next((i for i, a in enumerate(angles) if a >= 45), None),
            'time_to_10': next((i for i, a in enumerate(angles) if a >= 10), None),
            'phase_durations': {
                'Π⁽⁰⁾': phases.count('Π⁽⁰⁾'),
                'Π⁽¹⁾': phases.count('Π⁽¹⁾'),
                'Π⁽²⁾': phases.count('Π⁽²⁾'),
                'Π⁽³⁾': phases.count('Π⁽³⁾')
            }
        }

print("✓ ClaimEvolution class defined")

## 3. Scenario 1: Natural Drift to Collapse

A claim starts at 0° (VC/GOS) and gradually drifts until it collapses at 45°.

In [ ]:
# Scenario 1: Natural drift without recovery
print("=" * 60)
print("SCENARIO 1: Natural Drift to Collapse")
print("=" * 60)
print("\nA claim starts at 0° (VC/GOS) and drifts gradually.")
print("Watch what happens when it reaches 45°.\n")

sim1 = ClaimEvolution(initial_angle=0.0, drift_rate=1.5, noise_amplitude=0.5)
history1 = sim1.run(n_steps=40)

# Create visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Angle over time
ax1 = axes[0]
times = [h['time'] for h in history1]
angles = [h['angle'] for h in history1]
colors = [h['color'] for h in history1]

ax1.plot(times, angles, 'k-', linewidth=1.5, alpha=0.5)
ax1.scatter(times, angles, c=colors, s=30, zorder=5)

# Phase boundaries
ax1.axhline(10, color='#2ecc71', linestyle='--', alpha=0.7, label='Π⁽⁰⁾/Π⁽¹⁾ boundary (10°)')
ax1.axhline(35, color='#f1c40f', linestyle=':', alpha=0.7, label='Π⁽¹⁾/Π⁽²⁾ boundary (35°)')
ax1.axhline(45, color='#e74c3c', linestyle='--', linewidth=2, alpha=0.8, label='COLLAPSE BOUNDARY (45°)')
ax1.axhline(55, color='#95a5a6', linestyle=':', alpha=0.7, label='Π⁽²⁾/Π⁽³⁾ boundary (55°)')

# Shade zones
ax1.axhspan(0, 10, alpha=0.1, color='#2ecc71')
ax1.axhspan(10, 35, alpha=0.1, color='#f1c40f')
ax1.axhspan(35, 55, alpha=0.15, color='#e74c3c')
ax1.axhspan(55, 90, alpha=0.1, color='#95a5a6')

ax1.set_xlabel('Time Step')
ax1.set_ylabel('Angle (degrees)')
ax1.set_title('Angle Evolution: Natural Drift to Collapse')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Phase progression as a timeline
ax2 = axes[1]
phases = [h['phase'] for h in history1]
phase_colors = {'Π⁽⁰⁾': '#2ecc71', 'Π⁽¹⁾': '#f1c40f', 'Π⁽²⁾': '#e74c3c', 'Π⁽³⁾': '#95a5a6'}
phase_numeric = [{'Π⁽⁰⁾': 0, 'Π⁽¹⁾': 1, 'Π⁽²⁾': 2, 'Π⁽³⁾': 3}[p] for p in phases]

ax2.step(times, phase_numeric, where='mid', color='black', alpha=0.5)
ax2.scatter(times, phase_numeric, c=[phase_colors[p] for p in phases], s=50, zorder=5)
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(['Π⁽⁰⁾\nexpand', 'Π⁽¹⁾\nextend', 'Π⁽²⁾\nresist', 'Π⁽³⁾\nsynthesis'])
ax2.set_xlabel('Time Step')
ax2.set_title('Triplet Phase Progression')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Print summary
summary1 = sim1.get_summary()
print("\n" + "-" * 40)
print("SIMULATION SUMMARY")
print("-" * 40)
print(f"Final angle: {summary1['final_angle']:.1f}°")
print(f"Final phase: {summary1['final_phase']}")
print(f"Final status: {summary1['final_status']}")
print(f"Time to reach 10°: {summary1['time_to_10']} steps")
print(f"Time to reach 45° (collapse): {summary1['time_to_45']} steps")
print(f"\nPhase durations:")
for phase, duration in summary1['phase_durations'].items():
    print(f"  {phase}: {duration} steps")

## 4. Scenario 2: Drift, Collapse, and Recovery

A claim drifts, collapses at 45°, then recovers back to VC/GOS.

In [ ]:
# Scenario 2: Drift to collapse, then recovery
print("\n" + "=" * 60)
print("SCENARIO 2: Drift → Collapse → Recovery")
print("=" * 60)
print("\nA claim drifts, collapses at 45°, then recovers after intervention.")
print("Recovery is triggered at time step 25.\n")

sim2 = ClaimEvolution(initial_angle=0.0, drift_rate=2.0, noise_amplitude=0.8, recovery_rate=4.0)
history2 = sim2.run(n_steps=50, recovery_at=[25, 26, 27, 28, 29, 30])  # Force recovery for 6 steps

# Create visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Angle over time with recovery annotation
ax1 = axes[0]
times = [h['time'] for h in history2]
angles = [h['angle'] for h in history2]
colors = [h['color'] for h in history2]

ax1.plot(times, angles, 'k-', linewidth=1.5, alpha=0.5)
ax1.scatter(times, angles, c=colors, s=30, zorder=5)

# Phase boundaries
ax1.axhline(10, color='#2ecc71', linestyle='--', alpha=0.7)
ax1.axhline(35, color='#f1c40f', linestyle=':', alpha=0.7)
ax1.axhline(45, color='#e74c3c', linestyle='--', linewidth=2, alpha=0.8, label='COLLAPSE BOUNDARY')
ax1.axhline(55, color='#95a5a6', linestyle=':', alpha=0.7)

# Shade zones
ax1.axhspan(0, 10, alpha=0.1, color='#2ecc71')
ax1.axhspan(10, 35, alpha=0.1, color='#f1c40f')
ax1.axhspan(35, 55, alpha=0.15, color='#e74c3c')

# Highlight recovery region
ax1.axvspan(25, 30, alpha=0.2, color='blue', label='Recovery Intervention')

ax1.set_xlabel('Time Step')
ax1.set_ylabel('Angle (degrees)')
ax1.set_title('Angle Evolution: Collapse at 45° Followed by Recovery')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Phase progression
ax2 = axes[1]
phases = [h['phase'] for h in history2]
phase_numeric = [{'Π⁽⁰⁾': 0, 'Π⁽¹⁾': 1, 'Π⁽²⁾': 2, 'Π⁽³⁾': 3}[p] for p in phases]

ax2.step(times, phase_numeric, where='mid', color='black', alpha=0.5)
ax2.scatter(times, phase_numeric, c=[phase_colors[p] for p in phases], s=50, zorder=5)
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(['Π⁽⁰⁾\nexpand', 'Π⁽¹⁾\nextend', 'Π⁽²⁾\nresist', 'Π⁽³⁾\nsynthesis'])
ax2.set_xlabel('Time Step')
ax2.set_title('Triplet Phase Progression with Recovery')
ax2.axvspan(25, 30, alpha=0.2, color='blue')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Print summary
summary2 = sim2.get_summary()
print("\n" + "-" * 40)
print("SIMULATION SUMMARY")
print("-" * 40)
print(f"Final angle: {summary2['final_angle']:.1f}°")
print(f"Final phase: {summary2['final_phase']}")
print(f"Final status: {summary2['final_status']}")
print(f"Maximum angle reached: {summary2['max_angle']:.1f}°")
print(f"Time to reach 45°: {summary2['time_to_45']} steps")
print(f"\nRecovery successful: {summary2['final_angle'] < 10}")

## 5. Scenario 3: Multiple Collapse-Recovery Cycles

A claim that repeatedly drifts to collapse and recovers.

In [ ]:
# Scenario 3: Multiple cycles
print("\n" + "=" * 60)
print("SCENARIO 3: Multiple Collapse-Recovery Cycles")
print("=" * 60)
print("\nA claim repeatedly drifts to collapse and recovers.")
print("Recovery triggers at times 20, 45, 70.\n")

sim3 = ClaimEvolution(initial_angle=0.0, drift_rate=2.5, noise_amplitude=1.0, recovery_rate=5.0)
history3 = sim3.run(n_steps=90, recovery_at=[20, 21, 22, 23, 24, 45, 46, 47, 48, 49, 70, 71, 72, 73, 74])

# Create visualization
fig, ax = plt.subplots(figsize=(14, 6))

times = [h['time'] for h in history3]
angles = [h['angle'] for h in history3]
colors = [h['color'] for h in history3]

ax.plot(times, angles, 'k-', linewidth=1, alpha=0.4)
ax.scatter(times, angles, c=colors, s=20, zorder=5)

# Phase boundaries
ax.axhline(10, color='#2ecc71', linestyle='--', alpha=0.5)
ax.axhline(35, color='#f1c40f', linestyle=':', alpha=0.5)
ax.axhline(45, color='#e74c3c', linestyle='--', linewidth=2, alpha=0.7, label='45° COLLAPSE BOUNDARY')
ax.axhline(55, color='#95a5a6', linestyle=':', alpha=0.5)

# Shade critical zone
ax.axhspan(35, 55, alpha=0.1, color='#e74c3c')

# Mark recovery periods
recovery_periods = [(20, 25), (45, 50), (70, 75)]
for start, end in recovery_periods:
    ax.axvspan(start, end, alpha=0.15, color='blue')

ax.set_xlabel('Time Step')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Multiple Collapse-Recovery Cycles')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
summary3 = sim3.get_summary()
print("\n" + "-" * 40)
print("SIMULATION SUMMARY")
print("-" * 40)
print(f"Final angle: {summary3['final_angle']:.1f}°")
print(f"Final phase: {summary3['final_phase']}")
print(f"Maximum angle reached: {summary3['max_angle']:.1f}°")
print(f"\nPhase durations:")
for phase, duration in summary3['phase_durations'].items():
    print(f"  {phase}: {duration} steps")

## 6. Interactive Parameter Explorer

Adjust parameters and see how the claim evolves in real-time.

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

def run_interactive_simulation(drift_rate=2.0, noise=1.0, recovery_rate=4.0, 
                                n_steps=50, recovery_start=25, recovery_duration=6):
    """Run interactive simulation with user parameters."""
    clear_output(wait=True)
    
    # Create recovery time steps
    recovery_at = list(range(recovery_start, min(recovery_start + recovery_duration, n_steps)))
    
    sim = ClaimEvolution(initial_angle=0.0, drift_rate=drift_rate, 
                         noise_amplitude=noise, recovery_rate=recovery_rate)
    history = sim.run(n_steps=n_steps, recovery_at=recovery_at)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Angle evolution
    ax1 = axes[0]
    times = [h['time'] for h in history]
    angles = [h['angle'] for h in history]
    colors = [h['color'] for h in history]
    
    ax1.plot(times, angles, 'k-', linewidth=1.5, alpha=0.5)
    ax1.scatter(times, angles, c=colors, s=40, zorder=5)
    
    ax1.axhline(10, color='#2ecc71', linestyle='--', alpha=0.7)
    ax1.axhline(35, color='#f1c40f', linestyle=':', alpha=0.7)
    ax1.axhline(45, color='#e74c3c', linestyle='--', linewidth=2, alpha=0.8, label='45° COLLAPSE')
    ax1.axhline(55, color='#95a5a6', linestyle=':', alpha=0.7)
    
    ax1.axhspan(0, 10, alpha=0.1, color='#2ecc71')
    ax1.axhspan(10, 35, alpha=0.1, color='#f1c40f')
    ax1.axhspan(35, 55, alpha=0.15, color='#e74c3c')
    
    if recovery_at:
        ax1.axvspan(min(recovery_at), max(recovery_at)+1, alpha=0.2, color='blue', label='Recovery')
    
    ax1.set_xlabel('Time Step')
    ax1.set_ylabel('Angle (degrees)')
    ax1.set_title('Claim Evolution')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 90)
    
    # Plot 2: Phase pie chart
    ax2 = axes[1]
    phase_counts = sim.get_summary()['phase_durations']
    phases = list(phase_counts.keys())
    counts = list(phase_counts.values())
    phase_colors_list = ['#2ecc71', '#f1c40f', '#e74c3c', '#95a5a6']
    
    wedges, texts, autotexts = ax2.pie(counts, labels=phases, colors=phase_colors_list,
                                        autopct='%1.1f%%', startangle=90)
    ax2.set_title('Time Spent in Each Phase')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    summary = sim.get_summary()
    print(f"\n{'='*50}")
    print("SIMULATION RESULTS")
    print(f"{'='*50}")
    print(f"Final angle: {summary['final_angle']:.1f}°")
    print(f"Final status: {summary['final_status']}")
    print(f"Collapsed at 45°? {'YES' if summary['time_to_45'] else 'NO'}")
    if summary['time_to_45']:
        print(f"Time to collapse: {summary['time_to_45']} steps")

print("=" * 60)
print("INTERACTIVE PARAMETER EXPLORER")
print("=" * 60)
print("\nAdjust the parameters below to see how the claim evolves.\n")

interact(run_interactive_simulation,
         drift_rate=FloatSlider(min=0, max=5, step=0.2, value=2.0, description='Drift Rate (°/step)'),
         noise=FloatSlider(min=0, max=3, step=0.1, value=1.0, description='Noise Amplitude'),
         recovery_rate=FloatSlider(min=0, max=8, step=0.5, value=4.0, description='Recovery Rate (°/step)'),
         n_steps=IntSlider(min=20, max=100, step=5, value=50, description='Time Steps'),
         recovery_start=IntSlider(min=0, max=80, step=5, value=25, description='Recovery Start'),
         recovery_duration=IntSlider(min=0, max=20, step=1, value=6, description='Recovery Duration'))

## 7. Phase State Machine Visualization

A state machine diagram showing the triplet phase transitions.

In [ ]:
def show_state_machine():
    """Display the triplet phase state machine."""
    clear_output(wait=True)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')
    
    # Define states and positions
    states = {
        'Π⁽⁰⁾ expand': (0.2, 0.7),
        'Π⁽¹⁾ extend': (0.5, 0.7),
        'Π⁽²⁾ resist': (0.8, 0.7),
        'Π⁽³⁾ synthesis': (0.5, 0.3)
    }
    
    state_colors = {
        'Π⁽⁰⁾ expand': '#2ecc71',
        'Π⁽¹⁾ extend': '#f1c40f',
        'Π⁽²⁾ resist': '#e74c3c',
        'Π⁽³⁾ synthesis': '#95a5a6'
    }
    
    # Draw states
    for state, (x, y) in states.items():
        circle = Circle((x, y), 0.12, facecolor=state_colors[state], 
                        edgecolor='black', linewidth=2, alpha=0.8)
        ax.add_patch(circle)
        ax.text(x, y, state, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Draw transitions
    # Π⁽⁰⁾ → Π⁽¹⁾
    ax.annotate('', xy=(0.38, 0.7), xytext=(0.32, 0.7),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    ax.text(0.35, 0.73, 'drift', ha='center', fontsize=8)
    
    # Π⁽¹⁾ → Π⁽²⁾
    ax.annotate('', xy=(0.68, 0.7), xytext=(0.62, 0.7),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    ax.text(0.65, 0.73, 'drift', ha='center', fontsize=8)
    
    # Π⁽²⁾ → Π⁽³⁾
    ax.annotate('', xy=(0.72, 0.6), xytext=(0.7, 0.55),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    ax.text(0.78, 0.58, 'collapse\n(>55°)', ha='center', fontsize=8)
    
    # Π⁽³⁾ → Π⁽⁰⁾ (recovery)
    ax.annotate('', xy=(0.32, 0.5), xytext=(0.6, 0.38),
                arrowprops=dict(arrowstyle='->', lw=2, color='blue', 
                               connectionstyle='arc3,rad=0.3'))
    ax.text(0.45, 0.4, 'recovery\n(intervention)', ha='center', fontsize=8, color='blue')
    
    # Π⁽²⁾ → Π⁽⁰⁾ (direct recovery from critical zone)
    ax.annotate('', xy=(0.32, 0.65), xytext=(0.7, 0.65),
                arrowprops=dict(arrowstyle='->', lw=2, color='green',
                               connectionstyle='arc3,rad=-0.3'))
    ax.text(0.5, 0.62, 'early recovery\n(before 55°)', ha='center', fontsize=8, color='green')
    
    # Add angle boundaries
    ax.text(0.2, 0.82, '0°-10°', ha='center', fontsize=7, color='#2ecc71')
    ax.text(0.5, 0.82, '10°-35°', ha='center', fontsize=7, color='#f1c40f')
    ax.text(0.8, 0.82, '35°-55°', ha='center', fontsize=7, color='#e74c3c')
    ax.text(0.5, 0.18, '>55°', ha='center', fontsize=7, color='#95a5a6')
    
    # Critical annotation
    ax.annotate('⚠ CRITICAL BOUNDARY ⚠\n45° — Collapse Point', 
                xy=(0.8, 0.7), xytext=(0.85, 0.55),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                fontsize=9, color='red', ha='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('Triplet Phase State Machine', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

show_state_machine()

## 8. Animated Progression (Optional)

Creates an animation of the claim evolving through phases.

In [ ]:
from IPython.display import HTML

def create_animation(n_frames=50, drift_rate=2.0):
    """Create an animated visualization of claim evolution."""
    
    sim = ClaimEvolution(initial_angle=0.0, drift_rate=drift_rate, noise_amplitude=0.5)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Initialize plots
    angles_plot = np.linspace(0, 90, 100)
    stability = np.cos(np.radians(angles_plot))
    ax1.plot(angles_plot, stability, 'k-', linewidth=2)
    ax1.axvline(10, color='#2ecc71', linestyle='--', alpha=0.5)
    ax1.axvline(35, color='#f1c40f', linestyle=':', alpha=0.5)
    ax1.axvline(45, color='#e74c3c', linestyle='--', linewidth=2, alpha=0.7)
    ax1.axvline(55, color='#95a5a6', linestyle=':', alpha=0.5)
    ax1.fill_between(angles_plot, 0, stability, where=(angles_plot > 35) & (angles_plot < 55), 
                      color='#e74c3c', alpha=0.1)
    ax1.fill_between(angles_plot, 0, stability, where=angles_plot < 10, 
                      color='#2ecc71', alpha=0.1)
    ax1.set_xlim(0, 90)
    ax1.set_ylim(0, 1)
    ax1.set_xlabel('Angle (degrees)')
    ax1.set_ylabel('cos(θ)')
    ax1.set_title('Position on Cosine Curve')
    ax1.grid(True, alpha=0.3)
    
    point, = ax1.plot([], [], 'ro', markersize=12, zorder=5)
    
    # Phase gauge
    ax2.axis('off')
    ax2.set_xlim(-1.5, 1.5)
    ax2.set_ylim(-0.5, 1.5)
    ax2.set_title('Phase Status')
    
    status_text = ax2.text(0, 0.5, '', ha='center', va='center', fontsize=14, fontweight='bold')
    
    def update(frame):
        record = sim.step(frame)
        angle = record['angle']
        cos_val = np.cos(np.radians(angle))
        point.set_data([angle], [cos_val])
        point.set_color(record['color'])
        
        status_text.set_text(f"{record['phase']}\n{record['angle']:.1f}°\n{record['status']}")
        status_text.set_color(record['color'])
        
        return point, status_text
    
    from matplotlib.animation import FuncAnimation
    anim = FuncAnimation(fig, update, frames=n_frames, interval=100, blit=True, repeat=False)
    
    plt.close()
    return anim

print("\n" + "=" * 60)
print("ANIMATED PROGRESSION")
print("=" * 60)
print("\nRun the cell below to create an animation of the claim evolving.\n")

# Uncomment to run (takes a moment)
# anim = create_animation(n_frames=40, drift_rate=2.0)
# HTML(anim.to_jshtml())

print("To run the animation, uncomment the code in the cell above.")
print("Note: Animations work best in Jupyter locally; Colab may need ffmpeg.")

## 9. Summary: Triplet Progression Complete

### Phase Summary Table

| Phase | Angle Range | Status | Behavior | Signal:Noise |
|-------|-------------|--------|----------|--------------|
| **Π⁽⁰⁾ expand** | 0°-10° | VC/GOS (locked) | Stable expansion within constraints | Signal >> Noise |
| **Π⁽¹⁾ extend** | 10°-35° | weak VC (drifting) | Extending beyond core constraints | Signal > Noise |
| **Π⁽²⁾ resist** | 35°-55° | IA/ZOS (critical) | Resisting constraints → COLLAPSE at 45° | Signal ≈ Noise |
| **Π⁽³⁾ synthesis** | >55° | broken/decoupled | Disconnected from constraints | Noise > Signal |

### Key Insights from Simulations

1. **Natural drift** inevitably leads to collapse at 45° without intervention
2. **Recovery is possible** but requires active intervention (re-anchoring constraints)
3. **Early recovery** (before 45°) is more effective than late recovery
4. **Multiple cycles** show that claims can oscillate between phases
5. **Noise can accelerate or delay** the transition to collapse

### Connection to Your PCSP Seminar

The speaker's claim about Mal'cev algebras starts at **Π⁽⁰⁾ (expand)** — anchored to known CSP dichotomy.

If they were to over-extrapolate ("all finite algebras have a dichotomy"), they would drift into **Π⁽¹⁾ (extend)** and eventually **Π⁽²⁾ (resist)** where the claim collapses at 45°.

By explicitly acknowledging the open general case, they maintain **Π⁽⁰⁾ stability**.

### Complete Notebook Series

| Notebook | Focus |
|----------|-------|
| 00 | Geometric foundation |
| 01 | Core cosine constraint functions |
| 02 | Math claims (PCSP) hydration |
| 03 | Equation consistency (IA vs VC) |
| 04 | ML embedding alignment (signal vs noise) |
| 05 | Agent consistency (belief-action) |
| 06 | Interactive phase lock demo |
| **07** | **Triplet progression simulation (this notebook)** |

---

**Thank you for completing the Cosine Constraint Lab!** 🔬

Remember: **Stay locked below 10°. At 45°, collapse is inevitable.** 